# Day 12 Project Solution: Searchable Knowledge Base

All five Day 12 concepts composed into one working artifact:

| Lesson | Concept |
|--------|---------|
| 1 | Motivation: O(n) brute-force search vs ANN |
| 2 | `PersistentClient`, `get_or_create_collection`, `add`/`query` |
| 3 | `where`-clause metadata filters (`$eq`, `$in`) |
| 4 | `OllamaEmbeddingFunction` — text in, text out, no manual embedding |
| 5 | `upsert` (idempotent ingestion), `get` (inspection / CRUD audit) |

## Cell 1 — Imports and configuration

In [ ]:
import os
import urllib.request  # Python stdlib HTTP client — used here as a lightweight
                       # Ollama connectivity test (no extra dependencies);
                       # distinct from the `requests` package taught in Day 3
import chromadb
import ollama

# ── Configuration ────────────────────────────────────────────────────────────
EMBED_MODEL  = "nomic-embed-text"   # free, runs locally via Ollama
STORE_PATH   = "./kb_store"         # Chroma writes SQLite files here
COLLECTION   = "knowledge_base"
DOCS_FOLDER  = "../sample_docs"     # five .txt files provided
TOP_K        = 3

# Guard: confirm Ollama is reachable before doing any work
try:
    urllib.request.urlopen("http://localhost:11434/api/tags", timeout=3)
except Exception as exc:
    raise RuntimeError(
        "Ollama is not running. Start it with `ollama serve` then retry."
    ) from exc

print("Ollama reachable. Configuration loaded.")

## Cell 2 — OllamaEmbeddingFunction  *(Lesson 4)*

Subclassing `chromadb.EmbeddingFunction` wires Ollama into Chroma so that
every `add` and `query` call embeds text automatically — no manual vector
management in application code.

In [ ]:
class OllamaEmbeddingFunction(chromadb.EmbeddingFunction):
    """
    Wraps Ollama's embeddings endpoint as a Chroma EmbeddingFunction.

    Chroma calls __call__ automatically on add(documents=...) and
    query(query_texts=...) — the caller never touches a raw vector.

    Args:
        model: Ollama embedding model name (default: "nomic-embed-text").
    """

    def __init__(self, model: str = "nomic-embed-text") -> None:
        self.model = model

    def __call__(self, input: list[str]) -> list[list[float]]:
        # Ollama handles one string at a time, so we loop explicitly.
        # From Chroma's perspective this is a black box: strings in, vectors out.
        vectors = []
        for text in input:
            response = ollama.embeddings(model=self.model, prompt=text)
            vectors.append(list(response["embedding"]))
        return vectors


# Quick smoke-test: embed one string, confirm we get a non-empty vector
_ef = OllamaEmbeddingFunction(EMBED_MODEL)
_test_vec = _ef(["hello world"])[0]
assert len(_test_vec) > 0, "Embedding failed — got empty vector"
print(f"OllamaEmbeddingFunction OK — vector dim: {len(_test_vec)}")

## Cell 3 — Open the persistent collection  *(Lessons 2 + 4)*

`PersistentClient` writes SQLite files to `STORE_PATH`.
`get_or_create_collection` is safe to call on every run:
creates the collection the first time, reopens it on subsequent runs.
The `EmbeddingFunction` must be re-attached each session (it is a Python
object, not serialised to disk).

In [ ]:
def open_collection(
    store_path: str,
    collection_name: str,
    embed_model: str,
):
    """
    Factory: open (or create) a persistent Chroma collection.

    Centralising this in one function means every script that needs the
    collection calls open_collection() — the embedding function re-attachment
    is never forgotten.
    """
    ef     = OllamaEmbeddingFunction(model=embed_model)
    client = chromadb.PersistentClient(path=store_path)
    return client.get_or_create_collection(
        name=collection_name,
        embedding_function=ef,
    )


collection = open_collection(STORE_PATH, COLLECTION, EMBED_MODEL)
print(f"Collection '{collection.name}' open. Documents indexed: {collection.count()}")

## Cell 4 — Ingest sample_docs  *(Lessons 2 + 5)*

We use `upsert` instead of `add` so this cell is **idempotent**: running it
again (after the store already contains data) overwrites with identical content
rather than crashing on duplicate IDs.

Metadata stored per document:
- `filename` — basename of the source file (used for display)
- `category` — prefix before the first `_` in the filename (used for filtering)

In [ ]:
def ingest_folder(collection, folder: str) -> int:
    """
    Upsert every .txt file in folder into the collection.

    Uses upsert (not add) so the function is idempotent — re-runs are safe.
    No embeddings= argument: OllamaEmbeddingFunction handles that automatically.

    Returns:
        Number of files processed.
    """
    txt_files = sorted(
        f for f in os.listdir(folder) if f.endswith(".txt")
    )
    if not txt_files:
        print(f"No .txt files found in {folder}")
        return 0

    ids, documents, metadatas = [], [], []

    for fname in txt_files:
        path    = os.path.join(folder, fname)
        stem    = os.path.splitext(fname)[0]          # e.g. "ml_embeddings"
        # category = everything before the first underscore
        category = stem.split("_")[0]                 # e.g. "ml"

        with open(path, "r", encoding="utf-8") as fh:
            text = fh.read().strip()

        ids.append(stem)
        documents.append(text)
        metadatas.append({"filename": fname, "category": category})
        print(f"  Queued: {fname}  (category={category})")

    # upsert in one batch — OllamaEmbeddingFunction embeds each document
    collection.upsert(
        ids=ids,
        documents=documents,
        metadatas=metadatas,
    )
    return len(ids)


ingested = ingest_folder(collection, DOCS_FOLDER)
print(f"\nIngested {ingested} files. Total in collection: {collection.count()}")

## Cell 5 — Search function  *(Lessons 2 + 3)*

`search_kb` uses `query_texts` (not `query_embeddings`) because the
embedding function is attached to the collection — Chroma embeds the query
string automatically.

The optional `where` parameter is forwarded straight to `collection.query`.
When provided, the metadata filter runs **before** the vector comparison,
narrowing the candidate pool to only the matching documents.

In [ ]:
def search_kb(
    collection,
    query: str,
    top_k: int = 3,
    where: dict | None = None,
) -> list[dict]:
    """
    Semantic search over the knowledge base with optional metadata filter.

    Args:
        collection: Open Chroma collection (with EmbeddingFunction attached).
        query:      Natural-language search query.
        top_k:      Number of results to return.
        where:      Optional Chroma where-clause dict, e.g. {"category": "ml"}.
                    Filter runs before vector search — only matching docs are ranked.

    Returns:
        List of result dicts with keys: id, filename, category, distance, excerpt.
    """
    # Build query kwargs — only add 'where' when it is provided
    kwargs: dict = {
        "query_texts": [query],   # Chroma embeds this via OllamaEmbeddingFunction
        "n_results":   top_k,
    }
    if where is not None:
        kwargs["where"] = where

    results = collection.query(**kwargs)

    # results[*][0] — index 0 because we sent one query (batch size = 1)
    hits = []
    for doc_id, text, dist, meta in zip(
        results["ids"][0],
        results["documents"][0],
        results["distances"][0],
        results["metadatas"][0],
    ):
        hits.append({
            "id":       doc_id,
            "filename": meta.get("filename", doc_id),
            "category": meta.get("category", ""),
            "distance": dist,
            "excerpt":  text,
        })
    return hits


def print_results(results: list[dict]) -> None:
    """Display search results in a readable ranked list."""
    if not results:
        print("  (no results)")
        return
    for rank, hit in enumerate(results, start=1):
        print(f"\n  [{rank}] {hit['filename']}  (category={hit['category']}, "
              f"distance={hit['distance']:.4f})")
        excerpt = hit["excerpt"][:220].replace("\n", " ")
        print(f"      {excerpt}...")


print("search_kb and print_results defined.")

## Cell 6 — Smoke-test: two pre-canned queries

Before handing control to the interactive loop we run two fixed queries to
confirm the pipeline works end-to-end:

1. A broad semantic query (no filter) — demonstrates ANN retrieval.
2. The same query with a `$eq` category filter — demonstrates Lesson 3 filtering.

In [ ]:
SMOKE_QUERY = "how do databases store and find data efficiently?"

# ── Query 1: no filter — all docs are candidates ──────────────────────────
print(f'=== Unfiltered: "{SMOKE_QUERY}" ===')
results_all = search_kb(collection, SMOKE_QUERY, top_k=TOP_K)
print_results(results_all)

# ── Query 2: filter by category="db" — only database docs qualify ─────────
print(f'\n=== Filtered (category=db): "{SMOKE_QUERY}" ===')
results_db = search_kb(
    collection,
    SMOKE_QUERY,
    top_k=TOP_K,
    where={"category": {"$eq": "db"}},   # $eq filter — Lesson 3
)
print_results(results_db)

# Sanity assertions — do not change
assert len(results_all) > 0, "Expected at least one result for broad query"
assert all(h["category"] == "db" for h in results_db), \
    "Filtered results must all have category='db'"
print("\nSmoke-test assertions passed.")

## Cell 7 — Interactive search loop

Enter a natural-language query at the prompt. Optionally provide a category
filter (any of: `ml`, `db`, `python`). Empty query exits the loop.

**This is the deliverable.** Everything above was build-time; this is
query-time — and it is fast because the ANN index was built at upsert time.

In [ ]:
# In production this would be an interactive loop; for automated execution
# we run three pre-canned queries that demonstrate the same code paths.
DEMO_QUERIES = [
    {"query": "how do databases store and find data efficiently?", "category": None},
    {"query": "what are embeddings and how do they work?",          "category": "ml"},
    {"query": "logging best practices in production systems",        "category": "python"},
]

print("Knowledge base ready. Running pre-canned demo queries.")
print(f"Categories available: ml, db, python\n")

for entry in DEMO_QUERIES:
    query    = entry["query"]
    category = entry["category"]
    where    = {"category": {"$eq": category}} if category else None
    label    = f'category={category}' if where else "no filter"
    print(f'Results for "{query}" ({label}):')
    results = search_kb(collection, query, top_k=TOP_K, where=where)
    print_results(results)
    print()

print("Demo queries complete.")


## Cell 8 — CRUD inspection: verify persistence  *(Lesson 5)*

`collection.get()` does a direct key-value lookup — no query vector needed.
This confirms that the data written at upsert time is actually on disk and
readable in the same (or a later) session.

In [ ]:
# Inspect the first two documents by their IDs (filename stems)
inspect_ids = ["db_indexing", "ml_embeddings"]
fetched = collection.get(ids=inspect_ids)

print("=== CRUD inspection via collection.get() ===")
for doc_id, text, meta in zip(
    fetched["ids"],
    fetched["documents"],
    fetched["metadatas"],
):
    print(f"\nID:       {doc_id}")
    print(f"Filename: {meta['filename']}")
    print(f"Category: {meta['category']}")
    print(f"Excerpt:  {text[:120].replace(chr(10), ' ')}...")

# Confirm both IDs are present
assert len(fetched["ids"]) == 2, f"Expected 2 docs, got {len(fetched['ids'])}"
print("\nCRUD inspection passed — data persisted correctly.")

## Cell 9 — Upsert demonstration: idempotent re-ingestion  *(Lesson 5)*

Re-upserting an existing document overwrites it cleanly — the count stays
the same and no duplicate-ID error is raised. This is the correct pattern
for nightly or continuous ingestion pipelines.

In [ ]:
count_before = collection.count()

# Re-upsert one existing document with updated metadata
collection.upsert(
    ids=["python_logging"],
    documents=["Python's logging module: the standard for production-grade log management."],
    metadatas=[{"filename": "python_logging.txt", "category": "python", "updated": True}],
)

count_after = collection.count()
assert count_after == count_before, (
    f"Upsert should not increase count: before={count_before}, after={count_after}"
)

# Verify the new metadata is stored
record = collection.get(ids=["python_logging"])
assert record["metadatas"][0].get("updated") is True, \
    "Expected 'updated' flag in metadata after upsert"

print(f"Upsert OK — collection size unchanged ({count_after}), metadata updated.")

## Cell 10 — Final confirmation

In [ ]:
print("=" * 60)
print("Day 12 Project: Searchable Knowledge Base — COMPLETE")
print("=" * 60)
print(f"  Store path    : {STORE_PATH}  (SQLite, persists between runs)")
print(f"  Collection    : {COLLECTION}")
print(f"  Documents     : {collection.count()}")
print(f"  Embed model   : {EMBED_MODEL} (via OllamaEmbeddingFunction)")
print()
print("  Concepts demonstrated:")
print("    [1] O(n) scaling problem — motivation for ANN vector databases")
print("    [2] PersistentClient + get_or_create_collection + query")
print("    [3] where-clause metadata filters ($eq applied in smoke-test)")
print("    [4] OllamaEmbeddingFunction — raw text in, ranked text out")
print("    [5] upsert (idempotent ingestion) + get (CRUD inspection)")
print()
print("Deliverable produced: a local knowledge base queryable in natural")
print("language, persisted to disk, re-runs skip re-embedding.")